# Teste técnico do TCC — LLM no Colab, v2

Compara 3 formas de obter um escore do LLM e mede se a confiança separa acertos de erros.

1. **Binário A×B**: prompt 0/1, como no v1.
2. **Binário com ordem trocada**: média do log-odds de A×B e de B×A.
3. **Três opções**: 1 = mesmo, 0 = diferente, 2 = não dá para saber. Mostra p(2) como sinal de dúvida.

**Antes:** Ambiente de execução → Alterar o tipo → **T4 GPU**. Depois, **Executar tudo**.
Para testar o 7B, troque `USAR_7B = False` para `True` na célula de configuração e rode tudo de novo.

Os pares daqui são de desenvolvimento: servem para ajustar o prompt e **não podem entrar no conjunto de teste do TCC**.

In [ ]:
USAR_7B = False   # True = Qwen2.5-7B em 4 bits; False = Qwen2.5-3B em float16
REPETICOES_TEMPO = 1

In [ ]:
!pip -q install bitsandbytes
import torch, time, math, statistics, csv, platform
import transformers
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (rodando na CPU)")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| python", platform.python_version())

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

DEV = "cuda" if torch.cuda.is_available() else "cpu"
if USAR_7B:
    assert DEV == "cuda", "O 7B em 4 bits precisa da GPU"
    MODELO = "Qwen/Qwen2.5-7B-Instruct"
    kwargs = {"quantization_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                                         bnb_4bit_quant_type="nf4"),
              "device_map": "auto"}
else:
    MODELO = "Qwen/Qwen2.5-3B-Instruct"
    kwargs = {"torch_dtype": torch.float16 if DEV == "cuda" else torch.float32}

t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(MODELO)
model = AutoModelForCausalLM.from_pretrained(MODELO, **kwargs)
if not USAR_7B:
    model = model.to(DEV)
model.eval()
print(f"{MODELO} carregado em {time.perf_counter() - t0:.0f}s")

IDS = {}
for d in "012":
    t = tok.encode(d, add_special_tokens=False)
    assert len(t) == 1, f"'{d}' não é token único"
    IDS[d] = t[0]

In [ ]:
REGRA = ("Duas descrições são o MESMO produto somente se tiverem a mesma marca, a mesma variante "
         "(tipo, sabor, versão) e o mesmo conteúdo (peso ou volume). ")
BASE = "Você compara descrições de produtos de cupons fiscais de supermercados brasileiros. " + REGRA
PROMPT_BIN = BASE + "Responda apenas com um dígito: 1 se forem o mesmo produto, 0 se forem produtos diferentes."
PROMPT_TRI = BASE + ("Responda apenas com um dígito: 1 se forem o mesmo produto, 0 se forem produtos diferentes, "
                     "2 se as descrições não tiverem informação suficiente para decidir.")

# (A, B, verdade, decidível pelo texto)  verdade: 1 mesmo / 0 diferente
PARES = [
    ("FERM.ROYAL", "FERMENTO ROYAL 250G", 1, False),
    ("FERM PO ROYAL 250G", "FERMENTO EM P ROYAL 250G", 1, True),
    ("FERMENTO PO ROYAL 250G - POTE 250G", "Fermento Royal 250g", 1, True),
    ("BISC RECH MOCA 140G NESTLE", "BIS REC NESTLE MOCA 140G", 1, True),
    ("MOCA Biscoito Recheado 140g", "BISCOITO RECH. MOCA 140G", 1, True),
    ("LEITE COND. MOCA SEMIDESN. TP 395G", "LEITE COND SEMID MOCA TRAD TP 395G", 1, True),
    ("AG CRYSTAL C/G 1,5L", "AG CRYSTAL S/G 1,5L", 0, True),
    ("LEITE IN XANDO LIGHT GFA 1LT", "LEITE IN XANDO TIPO A GFA 1LT", 0, True),
    ("ATUM 88 SOL NAT 140G", "ATUM 88 SOLI 140G", 0, True),
    ("PE DE MOCA AMENDOIM E LEITE CONDENSADO POPULAR 55G", "LEITE COND MOCA", 0, True),
    ("MILHO PIP KISAB 400G", "FAR MILHO KISABOR PURO AMAR500", 0, True),
    ("PAO FORMA PANCO", "PAO BISNAG PANCO 300G", 0, True),
    ("L COND PIRACANJ 395G", "QJO PIRACANJUBA 500G MUSSA", 0, True),
    ("LEITE LV TIROL INTEG EDGE 1L", "LEITE IN XANDO TIPO A GFA 1LT", 0, True),
    ("MOLHO BILLY E JACK 200G OR", "MOLHO BILLY E JACK 200G TA", 0, False),
]

def logprobs_primeiro_token(sistema, a, b):
    msgs = [{"role": "system", "content": sistema},
            {"role": "user", "content": f"Descrição A: {a}\nDescrição B: {b}\nResposta:"}]
    texto = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    entrada = tok(texto, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**entrada).logits[0, -1]
    lp = torch.log_softmax(logits.float(), dim=-1)
    return {d: lp[IDS[d]].item() for d in "012"}

def escores(a, b):
    t = time.perf_counter()
    ab = logprobs_primeiro_token(PROMPT_BIN, a, b)
    ba = logprobs_primeiro_token(PROMPT_BIN, b, a)
    tri = logprobs_primeiro_token(PROMPT_TRI, a, b)
    if DEV == "cuda":
        torch.cuda.synchronize()
    seg = time.perf_counter() - t
    lo_ab = ab["1"] - ab["0"]
    lo_ba = ba["1"] - ba["0"]
    p = {d: math.exp(tri[d]) for d in "012"}
    soma = sum(p.values())
    return {"logodds_AB": lo_ab, "logodds_BA": lo_ba, "logodds_media": (lo_ab + lo_ba) / 2,
            "tri_p1": p["1"] / soma, "tri_p0": p["0"] / soma, "tri_p2": p["2"] / soma,
            "tri_massa": soma, "segundos_3_chamadas": seg}

In [ ]:
escores("TESTE", "TESTE")  # aquecimento

linhas = []
print(f"{'verd':>4} {'dec':>3} | {'AB':>7} {'BA':>7} {'média':>7} | {'p1':>5} {'p0':>5} {'p2':>5} | par")
for a, b, verd, dec in PARES:
    r = escores(a, b)
    linhas.append({"modelo": MODELO, "descricao_a": a, "descricao_b": b, "verdade": verd,
                   "decidivel_texto": dec, **r})
    print(f"{verd:>4} {'sim' if dec else 'NÃO':>3} | {r['logodds_AB']:+7.2f} {r['logodds_BA']:+7.2f} "
          f"{r['logodds_media']:+7.2f} | {r['tri_p1']:.2f}  {r['tri_p0']:.2f}  {r['tri_p2']:.2f} | {a} × {b}")

def resumo(nome, chave):
    erros, acertos = [], []
    for l in linhas:
        if not l["decidivel_texto"]:
            continue
        pred = 1 if l[chave] > 0 else 0
        (acertos if pred == l["verdade"] else erros).append(abs(l[chave]))
    print(f"{nome:>14}: {len(acertos)} acertos (|lo| mediana {statistics.median(acertos) if acertos else float('nan'):.1f}) | "
          f"{len(erros)} erros (|lo| mediana {statistics.median(erros) if erros else float('nan'):.1f})")

print("\nNos pares decidíveis pelo texto, a confiança dos erros deveria ser MENOR que a dos acertos:")
resumo("A×B", "logodds_AB")
resumo("média A×B/B×A", "logodds_media")
nd = [l for l in linhas if not l["decidivel_texto"]]
dd = [l for l in linhas if l["decidivel_texto"]]
print(f"\nTrês opções: p(2) médio nos não decidíveis = {statistics.mean(l['tri_p2'] for l in nd):.2f} | "
      f"nos decidíveis = {statistics.mean(l['tri_p2'] for l in dd):.2f}  (queremos o primeiro bem maior)")
print(f"Massa mínima em 0/1/2: {min(l['tri_massa'] for l in linhas):.2f}")
print(f"Tempo por par (3 chamadas, {DEV}): mediana {statistics.median(l['segundos_3_chamadas'] for l in linhas):.3f}s")

nome = "resultado_v2_" + ("7B" if USAR_7B else "3B") + ".csv"
with open(nome, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(linhas[0].keys()))
    w.writeheader(); w.writerows(linhas)
print("Arquivo salvo:", nome)